In [ ]:
import sys
import os
from pathlib import Path
from typing import List

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch

# BenchMARL
from benchmarl.algorithms import (
    # full observation in critic
    MappoConfig,
    MaddpgConfig,
    MasacConfig, 
    # no full observation in critic
    IppoConfig,
    IddpgConfig,
    IsacConfig,
    # Discrete only
    #QmixConfig
)
from benchmarl.benchmark import Benchmark
from benchmarl.environments import VmasTask
from benchmarl.experiment import ExperimentConfig
from benchmarl.models.mlp import MlpConfig
from benchmarl.eval_results import load_and_merge_json_dicts, Plotting, get_raw_dict_from_multirun_folder

# VUMAS task for BenchMARL
from benchmarl.environments import UrbanEnvTask

# Ploting
from matplotlib import pyplot as plt

In [ ]:
# Define your source directory path
task = "uav_ue_los"
task = "navigate"


exp_dir = "test_los"
exp_dir = "experiments"
# exp_dir = "los"
experiments_dir = project_root / "outputs" / exp_dir
plot_dir = project_root / "outputs" / "plots" / exp_dir


raw_dict = get_raw_dict_from_multirun_folder(
    multirun_folder=experiments_dir
)
processed_data = Plotting.process_data(raw_dict)
(
    environment_comparison_matrix,
    sample_efficiency_matrix,
) = Plotting.create_matrices(processed_data, env_name="urbanmarl")



In [ ]:
performance_profile_figure = Plotting.performance_profile_figure(
    environment_comparison_matrix=environment_comparison_matrix
)

save_path = plot_dir / "performance_profile.pdf"
performance_profile_figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
Plotting.aggregate_scores?

In [ ]:
aggregate_scores, d, c = Plotting.aggregate_scores(
    environment_comparison_matrix=environment_comparison_matrix
)

save_path = plot_dir / "aggregate_scores.pdf"
aggregate_scores.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
type(processed_data)
processed_data['urbanmarl'].keys()

In [ ]:
processed_data['extra']['environment_list']['urbanmarl']

In [ ]:
environemnt_sample_efficiency_curves, _, _ = Plotting.environemnt_sample_efficiency_curves(
    sample_effeciency_matrix=sample_efficiency_matrix
)

save_path = plot_dir / "environemnt_sample_efficiency_curves.pdf"
environemnt_sample_efficiency_curves.set_title(f"Sample Efficiency", fontsize=16)
environemnt_sample_efficiency_curves.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
for task in processed_data['urbanmarl'].keys():
    task_sample_efficiency_curves = Plotting.task_sample_efficiency_curves(
        processed_data=processed_data, env="urbanmarl", task=task
    )
    #
    save_path = plot_dir / f"{task}_sample_efficiency_curves.pdf"
    task_sample_efficiency_curves.set_title(f"{task.upper()} Sample Efficiency", fontsize=16)
    task_sample_efficiency_curves.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
task_sample_efficiency_curves.set_title?

In [ ]:
task_sample_efficiency_curves = Plotting.task_sample_efficiency_curves(
    processed_data=processed_data, env="urbanmarl", task=task
)

save_path = plot_dir / f"{task}_sample_efficiency_curves.pdf"
task_sample_efficiency_curves.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
algorithms = [
    "mappo",
    # "ippo", 
    "maddpg", 
    # "iddpg",
    "masac",
    # "isac",  
]
algo_compare = []
for i, algo1 in enumerate(algorithms):
    for j, algo2 in enumerate(algorithms):
        if i < j:
            pair = [algo1, algo2]
            pair_ =[algo2, algo1]
            if pair not in algo_compare and pair_ not in algo_compare:
                algo_compare.append(pair)
                print([algo1, algo2])
probability_of_improvement = Plotting.probability_of_improvement(
    environment_comparison_matrix,
    algorithms_to_compare=algo_compare,
)

save_path = plot_dir / "probability_of_improvement_m.pdf"
probability_of_improvement.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
algorithms = [
    # "mappo",
    "ippo", 
    # "maddpg", 
    "iddpg",
    # "masac",
    "isac",  
]
algo_compare = []
for i, algo1 in enumerate(algorithms):
    for j, algo2 in enumerate(algorithms):
        if i < j:
            pair = [algo1, algo2]
            pair_ =[algo2, algo1]
            if pair not in algo_compare and pair_ not in algo_compare:
                algo_compare.append(pair)
                print([algo1, algo2])
probability_of_improvement = Plotting.probability_of_improvement(
    environment_comparison_matrix,
    algorithms_to_compare=algo_compare,
)

save_path = plot_dir / "probability_of_improvement_i.pdf"
probability_of_improvement.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)

In [ ]:
Plotting.probability_of_improvement(
    environment_comparison_matrix,
    algorithms_to_compare=[["maddpg", "mappo", "masac", "IPPO", "ISAC", "IDDPG"], ["maddpg", "mappo", "masac", "IPPO", "ISAC", "IDDPG"]],
)

In [ ]:
def inspect_dict(d, indent=0, max_depth=None, current_depth=0, show_values=False):
    """
    Recursively prints the keys (and optional values) of a nested dictionary.
    
    Parameters:
        d : dict or list
            The data structure to inspect.
        indent : int
            Current indentation level (spaces).
        max_depth : int or None
            Maximum depth to descend. None means unlimited.
        current_depth : int
            Current recursion depth (used internally).
        show_values : bool
            If True, prints values for leaf nodes (non-dict/non-list).
    """
    prefix = " " * indent
    if isinstance(d, dict):
        for key, value in d.items():
            print(f"{prefix}├── {key}")
            # Recurse if value is a dict/list and we haven't hit max_depth
            if isinstance(value, (dict, list)) and (max_depth is None or current_depth < max_depth):
                inspect_dict(value, indent + 4, max_depth, current_depth + 1, show_values)
            else:
                if show_values:
                    # Print a short representation of the leaf value
                    val_str = repr(value)
                    if len(val_str) > 60:
                        val_str = val_str[:57] + "..."
                    print(f"{prefix}    └── Value: {val_str}")
    elif isinstance(d, list):
        print(f"{prefix}├── [list with {len(d)} elements]")
        # Optionally inspect the first few elements if they are dicts
        if max_depth is None or current_depth < max_depth:
            for i, item in enumerate(d):
                if isinstance(item, (dict, list)):
                    print(f"{prefix}    ├── index {i}:")
                    inspect_dict(item, indent + 8, max_depth, current_depth + 1, show_values)
                else:
                    if show_values:
                        val_str = repr(item)
                        if len(val_str) > 60:
                            val_str = val_str[:57] + "..."
                        print(f"{prefix}    └── index {i}: {val_str}")
    else:
        # Base case: not a container
        if show_values:
            val_str = repr(d)
            if len(val_str) > 60:
                val_str = val_str[:57] + "..."
            print(f"{prefix}└── Value: {val_str}")

In [ ]:
# inspect_dict(processed_data)

In [ ]:
processed_data['extra']['metric_list']['urbanmarl']

In [ ]:
Plotting.METRIC_TO_PLOT = 'uav_return'
environemnt_sample_efficiency_curves = Plotting.environemnt_sample_efficiency_curves(
    sample_effeciency_matrix=sample_efficiency_matrix
)

# save_path = plot_dir / "environemnt_sample_efficiency_curves.pdf"
# environemnt_sample_efficiency_curves.figure.savefig(save_path, bbox_inches='tight', pad_inches=0.1)